In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau

# calculate reward
def compute_reward(iq, previous_iq, score, previous_score):
    iq_diff = iq - previous_iq
    score_diff = score - previous_score
    
    score_reward = np.log(1 + abs(score_diff))
    iq_reward = np.log(1 + abs(iq_diff))
    
    if iq_diff < 0:
        iq_reward = -iq_reward
    if score_diff < 0:
        score_reward = -score_reward
    
    reward = (iq_reward + score_reward) / 2
    return reward

# extract epoch, score and IQ
def extract_epoch_score_iq(log_file_path):
    epoch = 0  
    data = []  

    with open(log_file_path, 'r', encoding='utf-8') as file:
        for line in file:
            match = re.search(r'DEBUG:score: (\d+\.\d+), iq:(\d+)', line)
            if match:
                score = float(match.group(1))
                iq = int(match.group(2))
       
                if iq > 0:
                    data.append((epoch, score, iq))
                
                epoch += 1

    df = pd.DataFrame(data, columns=["Epoch", "Score", "IQ"])
    
    # reward 
    df['Previous_IQ'] = df['IQ'].shift(1)
    df['Previous_Score'] = df['Score'].shift(1)
    df['Reward'] = df.apply(lambda row: compute_reward(row['IQ'], row['Previous_IQ'], row['Score'], row['Previous_Score']), axis=1)
    
    df.dropna(inplace=True)
    
    return df

log_dir = '../log/kendal/'

log_files = [f for f in os.listdir(log_dir) if f.endswith('.log')]

epoch_rewards = {}
epoch_iqs = {}

for log_file in log_files:
    log_file_path = os.path.join(log_dir, log_file)
    epoch_score_iq_df = extract_epoch_score_iq(log_file_path)

    filtered_df = epoch_score_iq_df[epoch_score_iq_df["Epoch"] >= 50]

    filtered_df = filtered_df[filtered_df["Score"] >= 120]

    filtered_df['cum_reward'] = filtered_df['Reward'].cumsum()
    filtered_df['mean_reward'] = filtered_df['cum_reward'] / (filtered_df['Epoch'] + 1)
    
    filtered_df['cum_iq'] = filtered_df['IQ'].cumsum()
    filtered_df['mean_iq'] = filtered_df['cum_iq'] / (filtered_df['Epoch'] + 1)

    filtered_df = filtered_df.reset_index(drop=True)
    filtered_df['Epoch'] = filtered_df.index + 1

    for _, row in filtered_df.iterrows():
        epoch = row['Epoch']
        mean_reward = row['mean_reward']
        mean_iq = row['mean_iq']
        if epoch not in epoch_rewards:
            epoch_rewards[epoch] = []
        if epoch not in epoch_iqs:
            epoch_iqs[epoch] = []
        epoch_rewards[epoch].append(mean_reward)
        epoch_iqs[epoch].append(mean_iq)

epochs = range(1, 400)
max_rewards = [max(epoch_rewards[epoch]) for epoch in epochs]
min_rewards = [min(epoch_rewards[epoch]) for epoch in epochs]
avg_rewards = [np.mean(epoch_rewards[epoch]) for epoch in epochs]

max_iqs = [max(epoch_iqs[epoch]) for epoch in epochs]
min_iqs = [min(epoch_iqs[epoch]) for epoch in epochs]
avg_iqs = [np.mean(epoch_iqs[epoch]) for epoch in epochs]

plt.figure(figsize=(10, 6))
plt.plot(epochs, avg_rewards, label='Average Reward', color='blue')
plt.fill_between(epochs, min_rewards, max_rewards, color='blue', alpha=0.2)
plt.title('Accumulated Reward by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accumulated Reward')
plt.grid(True)
plt.savefig('accumulated_reward.pdf') 
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(epochs, avg_iqs, label='Average IQ', color='green')
plt.fill_between(epochs, min_iqs, max_iqs, color='green', alpha=0.2)
plt.title('Accumulated IQ by Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accumulated IQ')
plt.grid(True)
plt.savefig('accumulated_iq.pdf') 
plt.show()